# Apache Spark Deep Dive Demo

This notebook demonstrates advanced Apache Spark concepts including:
- Catalyst optimizer and Tungsten execution engine internals
- Memory management and performance tuning strategies
- Advanced shuffle mechanisms and partitioning strategies
- Production deployment patterns and real-world use cases
- Performance troubleshooting and optimization techniques

In [ ]:
# Import required libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.storagelevel import StorageLevel
from pyspark import SparkConf
import time
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

## 1. Spark Session Configuration for Performance Tuning

In [ ]:
# Create optimized Spark session with performance tuning
conf = SparkConf()
conf.set("spark.sql.adaptive.enabled", "true")
conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "10000")
conf.set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
conf.set("spark.sql.shuffle.partitions", "200")
conf.set("spark.sql.broadcastTimeout", "36000")
conf.set("spark.sql.execution.arrow.fallback.enabled", "true")

spark = SparkSession.builder \
    .appName("SparkDeepDiveDemo") \
    .master("spark://spark-master:7077") \
    .config(conf=conf) \
    .getOrCreate()

# Set log level to reduce noise
spark.sparkContext.setLogLevel("WARN")

print(f"Spark Version: {spark.version}")
print(f"Spark Master: {spark.sparkContext.master}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")

## 2. Load and Explore Data

In [ ]:
# Load CSV files with schema inference
sales_df = spark.read.option("header", "true").option("inferSchema", "true").csv("/home/jovyan/data/sales_data.csv")
customers_df = spark.read.option("header", "true").option("inferSchema", "true").csv("/home/jovyan/data/customers.csv")
products_df = spark.read.option("header", "true").option("inferSchema", "true").csv("/home/jovyan/data/products.csv")

# Display schemas
print("Sales Data Schema:")
sales_df.printSchema()
print("\nCustomers Data Schema:")
customers_df.printSchema()
print("\nProducts Data Schema:")
products_df.printSchema()

# Show sample data
print("\nSample Sales Data:")
sales_df.show(5)
print("\nSample Customer Data:")
customers_df.show(5)
print("\nSample Product Data:")
products_df.show(5)

## 3. Catalyst Optimizer Demonstration

In [ ]:
# Register DataFrames as temporary views for SQL analysis
sales_df.createOrReplaceTempView("sales")
customers_df.createOrReplaceTempView("customers")
products_df.createOrReplaceTempView("products")

# Example 1: Predicate Pushdown Optimization
print("=== Catalyst Optimizer: Predicate Pushdown ===")

# Create a complex query that benefits from predicate pushdown
complex_query = """
SELECT 
    s.transaction_id,
    c.name as customer_name,
    p.name as product_name,
    s.amount,
    s.region
FROM sales s
JOIN customers c ON s.customer_id = c.customer_id
JOIN products p ON s.product_id = p.product_id
WHERE s.amount > 100 
    AND c.age > 30 
    AND p.category = 'Electronics'
"""

# Execute the query and show the logical plan
result_df = spark.sql(complex_query)
print("Logical Plan (optimized by Catalyst):")
result_df.explain(True)

print("\nQuery Results:")
result_df.show()

In [ ]:
# Example 2: Column Pruning and Projection Pushdown
print("=== Catalyst Optimizer: Column Pruning ===")

# Query that only selects specific columns
pruned_query = spark.sql("""
SELECT customer_id, amount, region
FROM sales
WHERE amount > 500
""")

print("Optimized plan with column pruning:")
pruned_query.explain(True)
pruned_query.show()

## 4. Tungsten Execution Engine and Code Generation

In [ ]:
# Example demonstrating Tungsten's code generation
print("=== Tungsten Execution Engine: Code Generation ===")

# Create a computation-heavy operation that benefits from code generation
tungsten_demo = sales_df.select(
    col("transaction_id"),
    col("amount"),
    (col("amount") * 0.1).alias("tax"),
    (col("amount") * 1.1).alias("total_with_tax"),
    when(col("amount") > 500, "High Value")
    .when(col("amount") > 100, "Medium Value")
    .otherwise("Low Value").alias("value_category")
).filter(col("amount") > 50)

print("Physical plan showing code generation:")
tungsten_demo.explain(mode="formatted")

print("\nResults:")
tungsten_demo.show()

## 5. Memory Management and Caching Strategies

In [ ]:
print("=== Memory Management and Caching Strategies ===")

# Create a large DataFrame for memory management demonstration
# Generate synthetic data to simulate larger datasets
large_sales_data = []
for i in range(10000):
    large_sales_data.append((
        i + 1000,  # transaction_id
        (i % 1000) + 1,  # customer_id
        (i % 100) + 1,  # product_id
        ["Electronics", "Clothing", "Books", "Sports"][i % 4],  # category
        round(np.random.uniform(10, 2000), 2),  # amount
        f"2024-01-{(i % 30) + 1:02d} {(i % 24):02d}:00:00",  # timestamp
        ["North", "South", "East", "West"][i % 4]  # region
    ))

large_sales_schema = StructType([
    StructField("transaction_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("category", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("timestamp", StringType(), True),
    StructField("region", StringType(), True)
])

large_sales_df = spark.createDataFrame(large_sales_data, large_sales_schema)

print(f"Large sales dataset count: {large_sales_df.count()}")

# Demonstrate different storage levels
print("\n1. Caching with MEMORY_ONLY:")
large_sales_df.cache()
start_time = time.time()
result1 = large_sales_df.groupBy("category").agg(sum("amount").alias("total_amount")).collect()
memory_only_time = time.time() - start_time
print(f"Execution time with MEMORY_ONLY: {memory_only_time:.3f} seconds")

# Unpersist and try with MEMORY_AND_DISK
large_sales_df.unpersist()
print("\n2. Caching with MEMORY_AND_DISK:")
large_sales_df.persist(StorageLevel.MEMORY_AND_DISK)
start_time = time.time()
result2 = large_sales_df.groupBy("category").agg(sum("amount").alias("total_amount")).collect()
memory_disk_time = time.time() - start_time
print(f"Execution time with MEMORY_AND_DISK: {memory_disk_time:.3f} seconds")

# Show storage information
print("\nStorage information:")
for rdd_info in spark.sparkContext.statusTracker().getRDDInfos():
    if rdd_info.numPartitions > 0:
        print(f"RDD ID: {rdd_info.id}, Partitions: {rdd_info.numPartitions}, Storage Level: {rdd_info.storageLevel}")

## 6. Advanced Partitioning Strategies

In [ ]:
print("=== Advanced Partitioning Strategies ===")

# 1. Hash Partitioning
print("1. Hash Partitioning by customer_id:")
hash_partitioned = large_sales_df.repartition(8, "customer_id")
print(f"Number of partitions: {hash_partitioned.rdd.getNumPartitions()}")

# Show partition distribution
partition_counts = hash_partitioned.rdd.mapPartitionsWithIndex(
    lambda idx, iterator: [(idx, sum(1 for _ in iterator))]
).collect()
print("Partition distribution (partition_id, record_count):")
for partition_id, count in partition_counts:
    print(f"Partition {partition_id}: {count} records")

# 2. Range Partitioning
print("\n2. Range Partitioning by amount:")
range_partitioned = large_sales_df.repartitionByRange(6, "amount")
print(f"Number of partitions: {range_partitioned.rdd.getNumPartitions()}")

# Show range distribution
range_stats = range_partitioned.rdd.mapPartitionsWithIndex(
    lambda idx, iterator: [
        (idx, 
         sum(1 for _ in iterator),
         min([row.amount for row in iterator] or [0]),
         max([row.amount for row in iterator] or [0])
        )
    ]
).collect()

print("Range partition distribution (partition_id, count, min_amount, max_amount):")
for partition_id, count, min_amt, max_amt in range_stats:
    if count > 0:
        print(f"Partition {partition_id}: {count} records, amount range: {min_amt:.2f} - {max_amt:.2f}")

# 3. Custom Partitioning by Region
print("\n3. Custom Partitioning by region:")
region_partitioned = large_sales_df.repartition("region")

# Analyze partition efficiency
region_analysis = region_partitioned.groupBy("region").count().orderBy("count", ascending=False)
print("Records per region:")
region_analysis.show()

## 7. Shuffle Mechanisms and Optimization

In [ ]:
print("=== Shuffle Mechanisms and Optimization ===")

# Create DataFrames for join operations
sales_large = large_sales_df.filter(col("amount") > 100)
customers_broadcast = customers_df

# 1. Broadcast Join (eliminates shuffle for small table)
print("1. Broadcast Join Example:")
from pyspark.sql.functions import broadcast

broadcast_join = sales_large.join(
    broadcast(customers_broadcast), 
    "customer_id", 
    "inner"
)

print("Broadcast join execution plan:")
broadcast_join.explain()

# 2. Sort-Merge Join (default for large-large joins)
print("\n2. Sort-Merge Join Example:")
# Disable broadcast join to force sort-merge join
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

sort_merge_join = sales_large.join(
    customers_df, 
    "customer_id", 
    "inner"
)

print("Sort-merge join execution plan:")
sort_merge_join.explain()

# Reset broadcast threshold
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")  # 10MB

# 3. Shuffle optimization with pre-partitioning
print("\n3. Shuffle Optimization with Pre-partitioning:")

# Pre-partition both DataFrames on join key
sales_partitioned = large_sales_df.repartition("customer_id")
customers_partitioned = customers_df.repartition("customer_id")

# Cache the partitioned DataFrames
sales_partitioned.cache()
customers_partitioned.cache()

# Perform multiple operations to see the benefit
start_time = time.time()
optimized_join = sales_partitioned.join(customers_partitioned, "customer_id", "inner")
result_count = optimized_join.count()
optimized_time = time.time() - start_time

print(f"Optimized join completed in {optimized_time:.3f} seconds")
print(f"Result count: {result_count}")

## 8. Performance Monitoring and Troubleshooting

In [ ]:
print("=== Performance Monitoring and Troubleshooting ===")

# 1. Monitor Spark Context and Job Execution
def get_spark_metrics():
    status_tracker = spark.sparkContext.statusTracker()
    
    print("Current Spark Application Status:")
    print(f"Application ID: {spark.sparkContext.applicationId}")
    print(f"Application Name: {spark.sparkContext.appName}")
    print(f"Active Jobs: {len(status_tracker.getActiveJobIds())}")
    print(f"Active Stages: {len(status_tracker.getActiveStageIds())}")
    
    # Get executor information
    executor_infos = status_tracker.getExecutorInfos()
    print(f"\nExecutor Information:")
    for executor in executor_infos:
        print(f"Executor {executor.executorId}: {executor.totalCores} cores, "
              f"{executor.maxMemory / 1024 / 1024:.0f}MB max memory, "
              f"{executor.memoryUsed / 1024 / 1024:.0f}MB used")

get_spark_metrics()

# 2. Custom performance measurement function
def measure_performance(operation_name, operation_func):
    print(f"\n=== Measuring Performance: {operation_name} ===")
    
    start_time = time.time()
    result = operation_func()
    end_time = time.time()
    
    execution_time = end_time - start_time
    
    print(f"Execution time: {execution_time:.3f} seconds")
    
    # Get latest job information
    status_tracker = spark.sparkContext.statusTracker()
    job_ids = status_tracker.getActiveJobIds()
    
    return result, execution_time

# 3. Performance comparison: Different aggregation strategies
def slow_aggregation():
    return large_sales_df.groupBy("category", "region").agg(
        sum("amount").alias("total_sales"),
        avg("amount").alias("avg_sales"),
        count("*").alias("transaction_count")
    ).collect()

def optimized_aggregation():
    return large_sales_df.cache().groupBy("category", "region").agg(
        sum("amount").alias("total_sales"),
        avg("amount").alias("avg_sales"),
        count("*").alias("transaction_count")
    ).collect()

# Compare performance
slow_result, slow_time = measure_performance("Slow Aggregation", slow_aggregation)
optimized_result, optimized_time = measure_performance("Optimized Aggregation", optimized_aggregation)

print(f"\nPerformance Improvement: {slow_time / optimized_time:.2f}x faster with caching")

## 9. Production Deployment Patterns

In [ ]:
print("=== Production Deployment Patterns ===")

# 1. Data Quality Checks
def perform_data_quality_checks(df, table_name):
    print(f"\nData Quality Report for {table_name}:")
    print(f"Total Records: {df.count()}")
    
    # Check for null values
    null_counts = df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0]
    print("Null Value Counts:")
    for col_name, null_count in null_counts.asDict().items():
        if null_count > 0:
            print(f"  {col_name}: {null_count}")
    
    # Check for duplicates
    total_records = df.count()
    distinct_records = df.distinct().count()
    duplicates = total_records - distinct_records
    print(f"Duplicate Records: {duplicates}")
    
    return duplicates == 0 and all(count == 0 for count in null_counts.asDict().values())

# Run data quality checks
sales_quality = perform_data_quality_checks(sales_df, "Sales")
customers_quality = perform_data_quality_checks(customers_df, "Customers")
products_quality = perform_data_quality_checks(products_df, "Products")

# 2. ETL Pipeline Pattern
def etl_pipeline():
    print("\n=== ETL Pipeline Execution ===")
    
    # Extract
    print("1. Extract phase - Loading data...")
    raw_sales = sales_df
    raw_customers = customers_df
    raw_products = products_df
    
    # Transform
    print("2. Transform phase - Processing data...")
    
    # Join all datasets
    enriched_sales = raw_sales.join(raw_customers, "customer_id", "inner") \
                              .join(raw_products, "product_id", "inner")
    
    # Add derived columns
    transformed_sales = enriched_sales.select(
        col("transaction_id"),
        col("customer_id"),
        col("product_id"),
        col("category"),
        col("amount"),
        to_date(col("timestamp"), "yyyy-MM-dd HH:mm:ss").alias("transaction_date"),
        col("region"),
        col("name").alias("customer_name"),
        col("products.name").alias("product_name"),
        (col("amount") * 0.1).alias("tax_amount"),
        (col("amount") * 1.1).alias("total_with_tax"),
        when(col("amount") > 500, "Premium")
        .when(col("amount") > 100, "Standard")
        .otherwise("Basic").alias("customer_tier")
    )
    
    # Load
    print("3. Load phase - Saving processed data...")
    
    # Write to PostgreSQL (simulated)
    postgres_properties = {
        "user": "spark",
        "password": "sparkpassword",
        "driver": "org.postgresql.Driver"
    }
    
    try:
        transformed_sales.write \
            .format("jdbc") \
            .option("url", "jdbc:postgresql://spark-postgres:5432/sparkdb") \
            .option("dbtable", "processed_sales") \
            .option("user", postgres_properties["user"]) \
            .option("password", postgres_properties["password"]) \
            .option("driver", postgres_properties["driver"]) \
            .mode("overwrite") \
            .save()
        print("Data successfully loaded to PostgreSQL")
    except Exception as e:
        print(f"Note: PostgreSQL connection not available in demo: {str(e)}")
        print("In production, data would be saved to database")
    
    # Alternative: Save as Parquet for production
    output_path = "/home/jovyan/data/output/processed_sales"
    transformed_sales.coalesce(1).write.mode("overwrite").parquet(output_path)
    print(f"Data saved as Parquet to {output_path}")
    
    return transformed_sales

# Execute ETL pipeline
processed_data = etl_pipeline()
print(f"\nProcessed {processed_data.count()} records")
processed_data.show(5)

## 10. Advanced Optimization Techniques

In [ ]:
print("=== Advanced Optimization Techniques ===")

# 1. Adaptive Query Execution (AQE) Demo
print("1. Adaptive Query Execution Analysis:")

# Enable AQE features
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

# Create a skewed dataset for demonstration
skewed_data = large_sales_df.filter(col("customer_id") <= 50)  # Most data in few customers
regular_data = large_sales_df.filter(col("customer_id") > 50)

skewed_join = skewed_data.join(customers_df, "customer_id", "inner")
print("AQE will optimize this skewed join:")
skewed_join.explain(mode="formatted")

# 2. Column Statistics and Cost-Based Optimization
print("\n2. Cost-Based Optimization with Statistics:")

# Analyze table statistics
large_sales_df.createOrReplaceTempView("large_sales")
spark.sql("ANALYZE TABLE large_sales COMPUTE STATISTICS")
spark.sql("ANALYZE TABLE large_sales COMPUTE STATISTICS FOR COLUMNS customer_id, amount, category")

# Show statistics
stats = spark.sql("DESCRIBE EXTENDED large_sales")
stats.filter(col("col_name").isin(["Statistics", "customer_id", "amount", "category"])).show(truncate=False)

# 3. Dynamic Partition Pruning
print("\n3. Dynamic Partition Pruning:")

# Create partitioned data
partitioned_sales = large_sales_df.repartition("region")
partitioned_sales.createOrReplaceTempView("partitioned_sales")

# Query with dynamic partition pruning
pruning_query = spark.sql("""
SELECT s.*, c.name
FROM partitioned_sales s
JOIN customers c ON s.customer_id = c.customer_id
WHERE s.region IN (SELECT DISTINCT region FROM partitioned_sales WHERE amount > 1000)
""")

print("Query plan with dynamic partition pruning:")
pruning_query.explain()

# 4. Memory and Storage Optimization
print("\n4. Memory and Storage Optimization:")

# Compare different storage formats
def test_storage_format(df, format_name, write_func):
    start_time = time.time()
    write_func()
    write_time = time.time() - start_time
    
    # Measure read performance
    start_time = time.time()
    read_df = spark.read.parquet(f"/home/jovyan/data/output/test_{format_name}")
    count = read_df.count()
    read_time = time.time() - start_time
    
    print(f"{format_name}: Write={write_time:.3f}s, Read={read_time:.3f}s, Count={count}")

# Test Parquet with different compression
test_data = large_sales_df.sample(0.1)  # Sample for faster testing

# Snappy compression (default)
test_storage_format(
    test_data, 
    "snappy",
    lambda: test_data.write.mode("overwrite").option("compression", "snappy").parquet("/home/jovyan/data/output/test_snappy")
)

# Gzip compression
test_storage_format(
    test_data, 
    "gzip",
    lambda: test_data.write.mode("overwrite").option("compression", "gzip").parquet("/home/jovyan/data/output/test_gzip")
)

## 11. Real-World Use Case: Streaming Analytics

In [ ]:
print("=== Real-World Use Case: Streaming Analytics Simulation ===")

# Simulate streaming data processing
from pyspark.sql.functions import window, current_timestamp

# Add timestamps to simulate streaming data
streaming_sales = large_sales_df.withColumn("event_time", current_timestamp())

# Windowed aggregations (simulating stream processing)
windowed_analytics = streaming_sales.groupBy(
    window(col("event_time"), "1 hour"),
    col("category"),
    col("region")
).agg(
    sum("amount").alias("hourly_revenue"),
    count("*").alias("transaction_count"),
    avg("amount").alias("avg_transaction_value"),
    max("amount").alias("max_transaction")
).orderBy("window")

print("Windowed analytics results:")
windowed_analytics.show(10, truncate=False)

# Real-time fraud detection pattern
fraud_detection = streaming_sales.withColumn(
    "fraud_score",
    when(col("amount") > 1500, 0.8)
    .when(col("amount") > 1000, 0.6)
    .when(col("amount") > 500, 0.3)
    .otherwise(0.1)
).filter(col("fraud_score") >= 0.6)

print(f"\nPotential fraud transactions detected: {fraud_detection.count()}")
fraud_detection.select("transaction_id", "customer_id", "amount", "fraud_score").show()

## 12. Performance Summary and Recommendations

In [ ]:
print("=== Performance Summary and Recommendations ===")

# Generate comprehensive performance report
def generate_performance_report():
    print("\n🚀 SPARK PERFORMANCE OPTIMIZATION SUMMARY")
    print("=" * 50)
    
    # Configuration recommendations
    print("\n📊 KEY CONFIGURATION SETTINGS:")
    important_configs = [
        "spark.sql.adaptive.enabled",
        "spark.sql.adaptive.coalescePartitions.enabled",
        "spark.sql.adaptive.skewJoin.enabled",
        "spark.sql.execution.arrow.pyspark.enabled",
        "spark.serializer"
    ]
    
    for config in important_configs:
        value = spark.conf.get(config, "Not Set")
        print(f"  {config}: {value}")
    
    # Resource utilization
    print("\n💾 RESOURCE UTILIZATION:")
    status_tracker = spark.sparkContext.statusTracker()
    executor_infos = status_tracker.getExecutorInfos()
    
    total_cores = sum(e.totalCores for e in executor_infos)
    total_memory = sum(e.maxMemory for e in executor_infos) / 1024 / 1024 / 1024  # GB
    used_memory = sum(e.memoryUsed for e in executor_infos) / 1024 / 1024 / 1024   # GB
    
    print(f"  Total Cores: {total_cores}")
    print(f"  Total Memory: {total_memory:.1f} GB")
    print(f"  Used Memory: {used_memory:.1f} GB ({used_memory/total_memory*100:.1f}%)")
    
    # Optimization recommendations
    print("\n🎯 OPTIMIZATION RECOMMENDATIONS:")
    recommendations = [
        "✅ Use Adaptive Query Execution (AQE) for dynamic optimization",
        "✅ Cache frequently accessed DataFrames with appropriate storage levels",
        "✅ Use broadcast joins for small tables (< 10MB)",
        "✅ Partition large datasets by frequently filtered columns",
        "✅ Use columnar formats (Parquet) for better compression and performance",
        "✅ Monitor and tune shuffle partitions based on data size",
        "✅ Use pushdown predicates to reduce data movement",
        "✅ Consider bucketing for frequently joined tables",
        "✅ Use column pruning to read only necessary columns",
        "✅ Monitor Spark UI for bottlenecks and skewed partitions"
    ]
    
    for rec in recommendations:
        print(f"  {rec}")
    
    print("\n🔧 PRODUCTION DEPLOYMENT CHECKLIST:")
    checklist = [
        "□ Set up proper monitoring and alerting",
        "□ Configure cluster auto-scaling",
        "□ Implement data quality checks",
        "□ Set up backup and disaster recovery",
        "□ Configure security and access controls",
        "□ Optimize for cost vs. performance trade-offs",
        "□ Test with production-scale data",
        "□ Document performance baselines"
    ]
    
    for item in checklist:
        print(f"  {item}")

generate_performance_report()

# Final cleanup
print("\n🧹 Cleaning up cached DataFrames...")
spark.catalog.clearCache()
print("Demo completed successfully! 🎉")